## Day 3 - Part 2: 자연어 처리 기초: 컴퓨터에게 말을 가르치다

### 개요

Day 3의 첫 파트에서 우리는 순서가 있는 데이터를 처리하는 강력한 도구인 RNN, LSTM, GRU를 배웠습니다. 

이를 통해 시계열 데이터의 패턴을 예측하는 데 성공했죠. 

이제 우리는 한 걸음 더 나아가, 순서가 있는 데이터의 가장 대표적이면서도 복잡한 형태인 `인간의 언어, 즉 자연어(Natural Language)`의 세계로 뛰어들 차례입니다.

"이 영화는 시간 가는 줄 모르고 봤어요. 인생 최고의 영화\!" 라는 문장과 "시간이 너무 아까웠어요. 제 인생 최악의 영화입니다." 라는 문장이 있다고 상상해 보세요. 두 문장은 비슷한 단어들('시간', '인생', '영화')을 포함하고 있지만, 그 의미는 정반대입니다. 이처럼 단어의 조합과 순서, 그리고 그 안에 담긴 미묘한 '감성'을 컴퓨터가 어떻게 이해하게 만들 수 있을까요?

이것이 바로 `자연어 처리(Natural Language Processing, NLP)` 의 핵심 질문입니다.  NLP는 컴퓨터가 인간의 언어를 이해하고, 해석하며, 생성할 수 있도록 만드는 인공지능의 한 분야입니다.  우리가 매일 사용하는 번역기, 챗봇, 스팸 메일 필터, 검색 엔진 등 수많은 기술이 NLP에 기반하고 있습니다. 

이번 파트에서는 '더러운' 원시 텍스트를 모델이 '먹을 수 있는' 깨끗한 재료로 손질하는 `전처리` 과정부터, 단어에 '의미'를 부여하는 `임베딩`, 그리고 이 모든 것을 종합하여 문장의 숨겨진 감성을 파악하는 `감성 분류 모델`을 구축하는 여정을 떠납니다. Part 1에서 배운 LSTM 모델이 어떻게 텍스트를 '읽고' 감정을 '느끼게' 되는지 직접 확인하게 될 것입니다.

`이번 파트의 학습 목표:`

  * 자연어 처리(NLP)의 개념과 중요성을 이해하고, 현실 속 적용 사례를 설명할 수 있습니다. 
  * 원시 텍스트를 모델 입력용으로 변환하는 `텍스트 전처리`의 필요성과 주요 기법(토큰화, 정제, 정수 인코딩, 패딩)을 이해하고 코드로 구현할 수 있습니다.
  * 단순 정수 인코딩의 한계를 설명하고, 단어의 의미를 벡터 공간에 표현하는 `워드 임베딩(Word Embedding)` 의 개념을 설명할 수 있습니다.
  * PyTorch의 `nn.Embedding` 레이어를 사용하여 텍스트 데이터를 딥러닝 모델에 입력하는 방법을 이해합니다.
  * RNN/LSTM 모델과 임베딩 레이어를 결합하여 `텍스트 분류 모델`을 설계하고 구현할 수 있습니다.
  * IMDB 영화 리뷰 데이터셋을 활용하여, 전처리부터 모델 훈련, 평가에 이르는 감성 분석 프로젝트 전 과정을 수행할 수 있습니다.

---
### 1. 왜 NLP가 필요할까?: 자연어 처리의 세계

자연어 처리(NLP)는 단순히 컴퓨터에게 단어를 가르치는 것을 넘어, 문맥과 의미, 심지어 감성까지 파악하게 하는 기술입니다.

  * `자연어 처리란?` 인간이 사용하는 언어(자연어)를 컴퓨터가 이해하고 처리할 수 있도록 하는 AI의 한 분야입니다. 목표는 인간과 컴퓨터 사이의 간극을 줄여 원활한 소통을 가능하게 하는 것입니다. 
  
  * `왜 중요할까?` 우리가 생성하는 데이터의 대부분은 비정형 텍스트 데이터입니다. 뉴스 기사, 이메일, 소셜 미디어 포스팅, 고객 리뷰 등에는 엄청난 양의 정보와 인사이트가 숨어있습니다.  NLP는 이 방대한 텍스트 더미 속에서 보물을 찾아내는 핵심 열쇠와도 같습니다.
      * `감성 분석:` 기업은 고객 리뷰를 분석하여 제품과 서비스에 대한 여론을 파악합니다. 
      
      * `기계 번역:` 구글 번역, 파파고 등은 언어의 장벽을 허물어 줍니다.
      * `정보 검색:` 검색 엔진은 우리의 검색 의도를 파악하여 가장 관련성 높은 문서를 찾아줍니다.
      * `챗봇 및 음성 비서:` Siri, Google Assistant 등은 우리의 말을 알아듣고 원하는 작업을 수행합니다. 

이 모든 마법 같은 일들의 첫걸음은, 뒤죽박죽인 텍스트를 기계가 이해할 수 있는 정돈된 형태로 바꾸는 것, 바로 '전처리'에서 시작됩니다.


---
### 2. 기계가 읽을 수 있는 텍스트: 전처리(Preprocessing)

모델에 텍스트를 입력하기 전에, "I love NLP\!", "nlp is fun.", "Amazing\! \<br\>" 와 같이 제멋대로인 텍스트를 일관성 있는 숫자 데이터로 변환하는 정제 과정이 반드시 필요합니다.  이를 `텍스트 전처리`라고 합니다. 주요 단계를 코드로 직접 확인하며 따라가 봅시다.

#### 2.1. 텍스트 정제 및 토큰화 (Cleaning & Tokenization)

가장 먼저 할 일은 분석에 불필요한 노이즈를 제거하고(정제), 문장을 최소 의미 단위인 토큰(Token)으로 나누는 것입니다(토큰화).

  * `정제(Cleaning):` 분석의 일관성을 위해 모든 알파벳을 소문자로 바꾸고, 분석에 방해가 되는 특수문자나 HTML 태그 등을 제거합니다. 
  
  * `토큰화(Tokenization):` 정제된 문장을 단어, 형태소 등 의미 있는 단위(토큰)로 쪼개는 과정입니다.  예를 들어 "I love NLP"는 `['i', 'love', 'nlp']` 라는 토큰 리스트가 됩니다.

<!-- end list -->

In [1]:
import re
import pandas as pd
from kiwipiepy import Kiwi

# 한글 예시 데이터
texts = ["정말 좋은 영화였어요!", "이 영화는 재미없어요.", "완전 최고! <br>", "정말 정말 좋은 영화이고 추천합니다."]

# Kiwi 형태소 분석기 초기화
kiwi = Kiwi()

# 통합된 전처리 함수: 정제와 형태소 분석을 한 번에 수행
def preprocess_text(text):
    # 1. 텍스트 정제: HTML 태그 제거 및 특수문자 정리
    text = re.sub(r'<[^>]+>', ' ', text)  # HTML 태그 제거
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', text)  # 한글, 영문, 숫자, 공백 외 문자 제거
    text = text.strip()
    
    # 2. Kiwi를 사용한 형태소 분석 및 토큰화
    tokens = kiwi.analyze(text)
    # 명사, 동사, 형용사 등 의미있는 형태소만 추출
    meaningful_tokens = []
    for token in tokens[0][0]:
        if token.tag in ['NNG', 'NNP', 'VV', 'VA', 'XR']:  # 명사, 동사, 형용사, 어근
            meaningful_tokens.append(token.form)
    
    return meaningful_tokens

# 모든 텍스트를 통합 전처리 함수로 처리
tokenized_texts = [preprocess_text(text) for text in texts]

print("토큰화된 텍스트:", tokenized_texts)

토큰화된 텍스트: [['좋', '영화'], ['영화', '재미없'], ['완전', '최고'], ['좋', '영화', '추천']]


#### 2.2. 정수 인코딩 (Integer Encoding)

컴퓨터는 'love'라는 단어 자체를 이해하지 못합니다. 대신 숫자로 바꿔주어야 합니다. 

이를 위해 전체 텍스트에 등장하는 모든 고유 단어의 집합인 `단어 사전(Vocabulary)`을 만들고, 각 단어에 고유한 정수(Index)를 부여합니다.

In [2]:
# 3. 단어 사전(Vocabulary) 구축 및 정수 인코딩
# tokenized_texts로부터 단어 사전 생성
vocab = sorted(list(set(word for tokens in tokenized_texts for word in tokens)))

# 각 단어에 고유한 정수 인덱스 부여
# 0번은 패딩(Padding)을 위해 비워두고 1번부터 시작
word_to_idx = {word: idx + 1 for idx, word in enumerate(vocab)}
print("Vocabulary:", word_to_idx)

# 4. 각 문장을 정수 시퀀스로 변환
encoded_sequences = [[word_to_idx[word] for word in tokens] for tokens in tokenized_texts]
print("Encoded Sequences:", encoded_sequences)

Vocabulary: {'영화': 1, '완전': 2, '재미없': 3, '좋': 4, '최고': 5, '추천': 6}
Encoded Sequences: [[4, 1], [1, 3], [2, 5], [4, 1, 6]]


이제 `['i', 'love', 'nlp']`는 `[5, 8, 9]`와 같이 컴퓨터가 처리할 수 있는 숫자 시퀀스로 변환되었습니다.

#### 2.3. 패딩 (Padding)

마지막 전처리 단계는 `패딩`입니다. 각 문장의 길이가 제각각이면(`[5, 8, 9]`는 길이 3, `[1]`은 길이 1) 행렬 연산 기반의 딥러닝 모델에 한 번에 여러 문장(배치)을 입력할 수 없습니다. 

따라서 모든 문장의 길이를 동일하게 맞춰주는 패딩 작업이 필요합니다. 보통 가장 긴 문장을 기준으로, 짧은 문장들의 뒤에 의미 없는 숫자 `0`을 채워 넣습니다.

In [3]:
# 5. 패딩: 모든 시퀀스의 길이를 동일하게 맞춤
max_len = max(len(seq) for seq in encoded_sequences)

padded_sequences = [seq + [0] * (max_len - len(seq)) for seq in encoded_sequences]

print("Padded Sequences (length={}):".format(max_len), padded_sequences)

Padded Sequences (length=3): [[4, 1, 0], [1, 3, 0], [2, 5, 0], [4, 1, 6]]


드디어 모든 문장이 동일한 길이의 정수 시퀀스로 변환되었습니다! 

이제 이 데이터를 딥러닝 모델의 입력으로 사용할 수 있습니다.



---
### 3. 단어에 의미를 불어넣다: 워드 임베딩(Word Embedding)

하지만 정수 인코딩에는 한 가지 치명적인 문제가 있습니다. 

`love=8`, `nlp=9`라고 할 때, 모델은 9가 8보다 크므로 'nlp'가 'love'보다 더 중요하거나 큰 값이라고 오해할 수 있습니다. 

숫자 자체에는 단어의 의미나 관계가 전혀 담겨있지 않습니다. 

`워드 임베딩`은 이 문제를 해결하기 위해 등장했습니다. 각 단어를, 그 의미를 압축한 저차원의 실수 벡터(Dense Vector)로 표현하는 기법입니다. 

  * `핵심 아이디어:` "비슷한 문맥에서 등장하는 단어는 비슷한 의미를 가질 것이다."
  
  * `결과:` '고양이'와 '강아지'는 벡터 공간에서 가까운 위치에, '책상'은 먼 위치에 표현됩니다. '왕' - '남자' + '여자' ≈ '여왕' 과 같은 의미적 연산도 가능해집니다.

PyTorch에서는 `nn.Embedding` 레이어를 통해 이 과정을 매우 간단하게 처리할 수 있습니다. 

이 레이어는 정수 인덱스를 입력받아, 그에 해당하는 임베딩 벡터를 출력하는 일종의 '룩업 테이블'입니다. 

이 테이블의 가중치(임베딩 벡터)들은 처음에는 랜덤값으로 시작하지만, 모델이 학습하는 과정에서 단어 간의 의미 관계를 스스로 학습하며 점차 정교하게 업데이트됩니다. 

#### 코드 실습: `nn.Embedding` 레이어 이해하기

In [4]:
import torch
import torch.nn as nn

# 전처리 완료된 패딩 시퀀스를 텐서로 변환
input_tensor = torch.LongTensor(padded_sequences)

# 하이퍼파라미터
vocab_size = len(word_to_idx) + 1  # 단어 사전의 크기 (+1 를 한것은 padding token '0'을 포함하기 위한 것)
print("vocab_size:", vocab_size)
embedding_dim = 8                 # 각 단어를 표현할 벡터의 차원 수

# 임베딩 레이어 정의
# padding_idx=0 : 0번 인덱스는 패딩 토큰이므로, 벡터를 0으로 채우고 학습시키지 않음
embedding_layer = nn.Embedding(num_embeddings=vocab_size,
                               embedding_dim=embedding_dim,
                               padding_idx=0)

# 임베딩 레이어 통과
embedding_output = embedding_layer(input_tensor)

print("Original Padded Tensor Shape:", input_tensor.shape)
print("Embedding Layer Output Shape:", embedding_output.shape)
# (배치 크기, 시퀀스 길이) -> (배치 크기, 시퀀스 길이, 임베딩 차원)

vocab_size: 7
Original Padded Tensor Shape: torch.Size([4, 3])
Embedding Layer Output Shape: torch.Size([4, 3, 8])


`[4, 7, 8]` 형태의 정수 시퀀스가 `[4, 7, 8]` 형태의 임베딩 벡터 시퀀스로 아름답게 변환되었습니다. 

이제 이 의미가 담긴 벡터 시퀀스를 Part 1에서 배운 LSTM 모델에 입력할 준비가 끝났습니다.

 ### 4. 종합 실습: 네이버 영화 리뷰 감성 분석
 
이제 배운 모든 것을 종합하여, `네이버 영화 리뷰 데이터셋(NSMC)`을 가지고 긍정/부정 감성 분석 모델을 만들어 보겠습니다. 
 
약 5만 개의 한국어 영화 리뷰 텍스트와 해당 리뷰가 긍정(1)인지 부정(0)인지 알려주는 레이블로 구성되어 있습니다.
 
#### 4.1. 데이터 준비 및 전처리
 
먼저, 로컬에 저장된 데이터를 불러와 살펴보고, 위에서 배운 전처리 과정을 전체 데이터에 적용하여 모델이 학습할 수 있는 형태로 가공합니다.

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import plotly.express as px

from tqdm import tqdm
tqdm.pandas()

# 1. 데이터 로드 (웹에서 직접 가져오기)
path = '../datasets/text/nsmc/ratings_train.txt'
df = pd.read_csv(path, sep='\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [6]:
# 2. 데이터 분포 확인
df['label'].value_counts()

label
0    75173
1    74827
Name: count, dtype: int64

In [7]:
df = df.dropna().sample(10000)

In [8]:
# 3. 텍스트 전처리 적용 (정제 및 토큰화)
df['tokenized_document'] = df['document'].progress_apply(preprocess_text)
df.head()

100%|██████████| 10000/10000 [00:06<00:00, 1506.19it/s]


,id,document,label,tokenized_document
114761,9223125,"주옥같은 대사들의 향연, 멋진 배경음악과 효과음, 출연배우들의 메소드급연기... 어...",0,"[주옥, 같, 대사, 향연, 멋지, 배경, 음악, 효과음, 출연, 배우, 메소드, ..."
80274,4449646,내면과 외면의 아름다움을 가지고있는 와리스디리의 감동적인 실화 꼭 봐야할 영화,1,"[내면, 외면, 아름다움, 가지, 와리스 디리, 감동, 실화, 보, 영화]"
62244,6561182,늦게 보게 되었지만 재미있습니다. 감동이였습니다.,1,"[늦, 보, 되, 재미있, 감동]"
59819,9652422,듣보잡영화의극치를달림,0,"[듣보잡, 영화, 극치, 달리]"
111068,9752424,영화제목만큼 유치하다,0,"[영화, 제목, 유치]"


In [9]:
tokenized_texts = df.tokenized_document.tolist()

In [10]:
# 4. 단어 사전 구축 및 정수 인코딩
from collections import Counter
word_counts = Counter(word for tokens in tokenized_texts for word in tokens) # 단어 : 개수 -> for문으로 돌면서 많은 순서대로 sorting -> listcom으로 받고 -> dictcom으로 다시 한 번 정수로 매핑
vocab_size = len(word_counts)
vocab = [word for word, count in word_counts.most_common(vocab_size-1)]
word_to_idx = {word: idx + 1 for idx, word in enumerate(vocab)}
word_to_idx['<unk>'] = 0 # 사전에 없는 단어(UNK)는 0번 인덱스 사용

In [11]:
# 정수 인코딩 (사전에 없으면 0으로 처리)
encoded_sequences = [[word_to_idx.get(word, 0) for word in tokens] for tokens in tokenized_texts]

In [12]:
# 적절한 max_len 구하기
# 시퀀스 길이 분포 확인
sequence_lengths = [len(seq) for seq in encoded_sequences]

# 기본 통계 정보
print(f"평균 길이: {np.mean(sequence_lengths):.2f}")
print(f"중앙값: {np.median(sequence_lengths):.2f}")
print(f"표준편차: {np.std(sequence_lengths):.2f}")
print(f"최소 길이: {min(sequence_lengths)}")
print(f"최대 길이: {max(sequence_lengths)}")

# 분위수 확인
percentiles = [50, 75, 90, 95, 99]
for p in percentiles:
    length = np.percentile(sequence_lengths, p)
    print(f"{p}% 분위수: {length:.2f}")

# 시각화
import plotly.graph_objects as go
import plotly.express as px

# 히스토그램
fig = px.histogram(x=sequence_lengths, nbins=50, 
                   title="시퀀스 길이 분포",
                   labels={'x': '시퀀스 길이', 'y': '빈도'})
fig.add_vline(x=np.mean(sequence_lengths), line_dash="dash", line_color="red", 
              annotation_text=f"평균: {np.mean(sequence_lengths):.1f}")
fig.add_vline(x=np.percentile(sequence_lengths, 95), line_dash="dash", line_color="orange",
              annotation_text=f"95%: {np.percentile(sequence_lengths, 95):.1f}")
fig.show()

# 추천 max_len 설정
recommended_max_len = int(np.percentile(sequence_lengths, 95))
print(f"추천 max_len: {recommended_max_len}")
print(f"이 길이로 설정하면 전체 데이터의 95%를 커버할 수 있습니다.")


평균 길이: 6.72
중앙값: 5.00
표준편차: 5.91
최소 길이: 0
최대 길이: 40
50% 분위수: 5.00
75% 분위수: 8.00
90% 분위수: 15.00
95% 분위수: 20.00
99% 분위수: 28.00


추천 max_len: 20
이 길이로 설정하면 전체 데이터의 95%를 커버할 수 있습니다.


In [13]:
# 5. 패딩
# 리뷰 길이가 매우 다양하므로, 최대 길이를 적절히 제한하는 것이 효율적
max_len = 20
padded_sequences = np.array([seq[:max_len] + [0]*(max_len - len(seq)) if len(seq) < max_len else seq[:max_len] for seq in encoded_sequences])

In [14]:
# 6. 훈련/테스트 데이터 분리 및 텐서 변환
X = padded_sequences
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# PyTorch 텐서로 변환
X_train_tensor = torch.LongTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.LongTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

# DataLoader 생성
from torch.utils.data import TensorDataset, DataLoader
batch_size = 64

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

#### 4.2. 감성 분석 모델 정의하기

이제 임베딩 레이어, LSTM 레이어, 그리고 최종 출력을 위한 완전연결층(Linear)을 결합하여 감성 분석 모델을 정의합니다.

In [15]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, drop_prob=0.5):
        super(SentimentLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        # 임베딩 레이어
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # LSTM 레이어
        self.lstm = nn.LSTM(embedding_dim,
                              hidden_dim,
                              n_layers,
                              dropout=drop_prob,
                              batch_first=True)

        # 드롭아웃 레이어
        self.dropout = nn.Dropout(drop_prob)

        # 완전연결층
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (batch_size, seq_length)

        # 임베딩
        embedded = self.embedding(x) # (batch_size, seq_length, embedding_dim)

        # LSTM
        # LSTM의 마지막 은닉 상태만 사용
        lstm_out, (hidden, cell) = self.lstm(embedded) # hidden: (n_layers, batch_size, hidden_dim)

        # 마지막 타임스텝의 은닉 상태를 사용
        # (batch_size, hidden_dim) 형태로 변환
        last_hidden = hidden[-1]

        # 드롭아웃 및 완전연결층
        out = self.dropout(last_hidden)
        out = self.fc(out)

        return out

# 모델 인스턴스 생성
embedding_dim = 64  
hidden_dim = 128    
output_dim = 2       # 긍정(1), 부정(0) -> 2개의 클래스
n_layers = 3         

# vocab_size는 10000으로 위에서 정의
model = SentimentLSTM(vocab_size, embedding_dim, hidden_dim, output_dim, n_layers)
print(model)

SentimentLSTM(
  (embedding): Embedding(9831, 64, padding_idx=0)
  (lstm): LSTM(64, 128, num_layers=3, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


#### 4.3. 모델 학습 및 평가

이진 분류 문제이므로 손실 함수는 `CrossEntropyLoss`를 사용하고, `Adam` 옵티마이저로 모델을 학습시킵니다.

In [18]:
import torch.optim as optim

# 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 모델 학습
num_epochs = 6 # 실제로는 더 많은 epoch가 필요할 수 있습니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("mps") # mac m1 이상 사용자
model.to(device)

for epoch in range(num_epochs):
    model.train() # 학습 모드
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

    # 에포크마다 테스트 데이터로 정확도 평가
    model.eval() # 평가 모드
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Test Accuracy: {accuracy:.2f}%')

print("훈련 종료!")

Epoch [1/6], Loss: 0.6964, Test Accuracy: 61.40%
Epoch [2/6], Loss: 0.5238, Test Accuracy: 72.00%
Epoch [3/6], Loss: 0.5583, Test Accuracy: 73.20%
Epoch [4/6], Loss: 0.4501, Test Accuracy: 74.20%
Epoch [5/6], Loss: 0.5828, Test Accuracy: 68.75%
Epoch [6/6], Loss: 0.3507, Test Accuracy: 74.95%
훈련 종료!


#### 4.4. 새로운 문장으로 예측해보기

학습이 완료된 모델을 사용하여, 직접 작성한 문장의 감성을 예측하는 함수를 만들어 봅시다. 

이 함수는 우리가 훈련 시 사용했던 전처리 과정을 그대로 거쳐야 합니다.

In [81]:
def predict_sentiment(text):
    model.eval()

    # 1. 텍스트 전처리
    tokenized = preprocess_text(text)

    # 2. 정수 인코딩
    encoded = [word_to_idx.get(word, 0) for word in tokenized]

    # 3. 패딩
    padded = np.array([encoded[:max_len] + [0]*(max_len - len(encoded)) if len(encoded) < max_len else encoded[:max_len]])

    # 4. 텐서 변환 및 디바이스 할당
    input_tensor = torch.LongTensor(padded).to(device)

    # 5. 예측
    with torch.no_grad():
        output = model(input_tensor)
        _, prediction = torch.max(output, 1)

    return "긍정적인 리뷰" if prediction.item() == 1 else "부정적인 리뷰"

# 테스트
test_review_1 = "이 영화는 정말 환상적이었어요! 연기가 뛰어나고 스토리가 매력적이었습니다."
test_review_2 = "시간 낭비였어요. 줄거리가 지루하고 예측 가능했어요."

print(f"리뷰 1: '{test_review_1}' -> 예측: {predict_sentiment(test_review_1)}")
print(f"리뷰 2: '{test_review_2}' -> 예측: {predict_sentiment(test_review_2)}")

리뷰 1: '이 영화는 정말 환상적이었어요! 연기가 뛰어나고 스토리가 매력적이었습니다.' -> 예측: 긍정적인 리뷰
리뷰 2: '시간 낭비였어요. 줄거리가 지루하고 예측 가능했어요.' -> 예측: 부정적인 리뷰


모델이 새로운 문장에 대해서도 꽤나 정확하게 감성을 예측하는 것을 볼 수 있습니다. 

이것으로 우리는 텍스트의 미묘한 맥락을 이해하고 분류하는 강력한 NLP 모델을 성공적으로 구축했습니다!